In [ ]:
# Libraries

import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from datetime import datetime
from tqdm import tqdm
import datetime as dtm

import matplotlib.pyplot as plt
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import cartopy.feature as cf

import xarray as xr

from scipy import stats

In [ ]:
# per capire come è costituito il dataset
flist = [os.path.join(path, name) 
         for path, subdirs, 
         files in os.walk("C:\\Users\\kekko\\Downloads\\Python\\Lab of Geo\\CRUTEM.4.6.0.0.station_files") 
         for name in files]  # contiene i dati grezzi derivanti da una serie di stazioni meteorologiche

for i in range(0,5):
  print (flist[i])

# exclude first 2 items, not relevant
flist=flist[2:]

print ('\n',flist[0:5])
nst=len(flist)
print ("\n > Number of stations = ",nst)  # si hanno più di 10k stazioni

## Aggiunta delle stazioni irlandesi

Per **aumentare il numero di stazioni** disponibili e migliorare la copertura
spaziale dell'area di studio, l'analisi viene estesa dalle sole stazioni del
**Regno Unito** anche a quelle dell'**Irlanda**. Le Isole Britanniche
costituiscono un'area climaticamente omogenea (clima oceanico temperato), per cui
trattarle insieme è coerente dal punto di vista geografico e statistico: più
stazioni significano una media regionale più robusta e meno sensibile ai valori
mancanti delle singole serie.

Concretamente le modifiche sono due:

* la **cella diagnostica** qui sotto è stata aggiornata per individuare *sia* le
  varianti della stringa paese per il Regno Unito *sia* quelle per l'Irlanda;
* il **filtro paese** nella cella di lettura dei dati accetta ora l'insieme
  `TARGET_COUNTRIES = {'UK', 'IRELAND'}` invece del solo `'UK'`.

In [ ]:
# --- VERIFICA STRINGHE PAESE (eseguire una sola volta, prima del loop completo) ---
# Il campo 'Country' nei file CRUTEM non e' standardizzato in modo univoco:
# a seconda della versione del dataset puo' comparire come 'UK', 'UNITED KINGDOM',
# 'IRELAND', 'IRISH REPUBLIC', ecc. Oltre al Regno Unito vogliamo ora includere
# anche l'Irlanda per aumentare il numero di stazioni disponibili. Qui si campiona
# un sottoinsieme di file per individuare le stringhe esatte (UK e Irlanda) da
# usare nel filtro della cella successiva.

found_countries = set()
for filein in flist[:3000]:  # campione, non l'intero dataset
    with open(filein, 'r') as f:
        for line in f:
            if line.startswith('Country'):
                found_countries.add(line.split('=', 1)[1].strip())
                break

uk_like = sorted(c for c in found_countries
                 if 'UK' in c.upper() or 'KINGDOM' in c.upper() or 'BRITAIN' in c.upper())
ie_like = sorted(c for c in found_countries
                 if 'IRELAND' in c.upper() or 'EIRE' in c.upper() or 'IRISH' in c.upper())
print('Possibili varianti trovate per UK:     ', uk_like)
print('Possibili varianti trovate per Irlanda:', ie_like)
print('\n> Copia le stringhe esatte nella variabile TARGET_COUNTRIES nella cella successiva.')

In [ ]:
# The first challenge is to correctly read in the data from an individual ASCII files.

# Define df to store metadata.
# Station IDs will be used as unique identifiers for both dataframes.

metadata0 = pd.DataFrame(columns=[
    'ID',
    'stname',
    'country',
    'elev',
    'lat',
    'lon'])

In [ ]:
# define a common time axis
taxis = pd.date_range('1850-01', '2019-01', freq='ME')
nmonths=len(taxis)
nyears=nmonths/12

# Define df to store data (temperature time series) and location. 
data0 = pd.DataFrame({'time': taxis})
data0 = data0.set_index(['time'])

# Define reference period
yref0 = 1951
yref1 = 2010

In [ ]:
# Liste per accumulare i dati (molto più veloce di append/merge in loop)
metadata_list = []
data_series_list = []

nodatacount = tooshortcount = outsidecount = 0

for filein in tqdm(flist[:nst]):
    with open(filein, 'r') as f:
        lines = f.readlines()

    # Estrazione rapida metadati in un colpo solo
    header = {}
    obs_index = -1
    for i, line in enumerate(lines):
        if '=' in line:
            k, v = line.split('=', 1)
            header[k.strip()] = v.strip()
        if "Obs:" in line:
            obs_index = i + 1
            break

    # 1. Filtro esistenza dati
    if obs_index == -1 or len(lines) <= obs_index:
        nodatacount += 1
        continue

    # 2. Filtro Country e Normals (rapidi)
    # NB: usare le stringhe esatte individuate con la cella diagnostica precedente.
    # Includiamo sia il Regno Unito sia l'Irlanda per aumentare il numero di stazioni.
    TARGET_COUNTRIES = {'UK', 'IRELAND'}  # <-- aggiornare in base all'output della cella diagnostica
    if header.get('Country', '').strip().upper() not in TARGET_COUNTRIES:
        continue
    
    if "-99.0" in header.get('Normals', ''):
        outsidecount += 1
        continue

    # Lettura dati numerici (solo una volta)
    try:
        raw_data = np.loadtxt(lines[obs_index:], dtype='f4')
        if raw_data.ndim == 1: raw_data = raw_data.reshape(1, -1) # Fix per file con 1 sola riga
    except:
        continue

    # 3. Filtro durata (anni)
    nyr = len(raw_data)
    if nyr < 30:
        tooshortcount += 1
        continue

    # Salvataggio Metadati
    st_id = filein[-6:]
    metadata_list.append({
        "ID": st_id, "nome": header.get('Name'), "paese": header.get('Country'),
        "altezza": float(header.get('Height', 0)), "lat": float(header.get('Lat', 0)), 
        "lon": float(header.get('Long', 0))
    })

    # Preparazione serie temporale
    # flatten() dei dati escludendo la prima colonna (l'anno)
    values = raw_data[:, 1:13].flatten()
    
    ti0 = datetime(int(raw_data[0, 0]), 1, 1)
    stime = pd.date_range(ti0, periods=len(values), freq='ME') # 'ME' è il nuovo standard per Month End
    
    data_series_list.append(pd.Series(values, index=stime, name=st_id))

# Creazione DataFrame finali (una sola operazione distruttiva alla fine)
metadata0 = pd.DataFrame(metadata_list)
data0 = pd.concat(data_series_list, axis=1)

In [ ]:
# --- Summary ---
print(f"Skipped - no data:     {nodatacount}")
print(f"Skipped - too short:   {tooshortcount}")
print(f"Skipped - outside ref: {outsidecount}")
print(f"Stations retained:     {len(metadata0)}")

In [ ]:
# Display metadata and data
print("\nMetadata:")
print(metadata0)

The CRUTEM dataset contains thousands of global stations, and applying filters (country = United Kingdom, at least 30 years of data, valid normals) yields a subset of UK stations. This ensures that the analyzed time series are long enough to be statistically representative of the local climate.

In [ ]:
# correzione della longitudine
metadata0.lon = -metadata0.lon

In [ ]:
# Visualizzazione stazione
lon, lat = metadata0.lon, metadata0.lat
pc = ccrs.PlateCarree()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4),
                                subplot_kw=dict(projection=pc))

# --- Vista globale ---
ax1.set_title("Vista globale")
ax1.stock_img()
ax1.coastlines()

# --- Zoom UK ---
ax2.set_title("Zoom Regno Unito e Irlanda")
ax2.set_extent([-11, 2, 49.5, 61], crs=pc)  # bounding box Isole Britanniche (UK + Irlanda)
ax2.coastlines(resolution='10m')
for feat, kw in [
    (cfeature.BORDERS, dict(linestyle=':')),
    (cfeature.LAND,   dict(facecolor='lightgray')),
    (cfeature.OCEAN,  dict(facecolor='lightblue')),
]:
    ax2.add_feature(feat, **kw)
ax2.gridlines(draw_labels=True, linestyle='--', alpha=0.5)

# scatter su entrambi gli assi
for ax, s in [(ax1, 10), (ax2, 50)]:
    ax.scatter([lon], [lat], color='r', s=s, transform=pc)

plt.tight_layout()
plt.show()

In [ ]:
# Sostituisce i -99 con NaN in tutto il DataFrame
# Tutte le colonne vengono mantenute, nessuna viene eliminata.
data0 = data0.replace(-99.0, np.nan)


In [ ]:
# Grafico combinato: mappa + serie temporale per ogni stazione
n = len(metadata0)
fig = plt.figure(figsize=(8, 2.5 * n))  

for i, (_, row) in enumerate(metadata0.iterrows()):
    # Mappa (sinistra)
    ax_map = plt.subplot(n, 2, i * 2 + 1, projection=ccrs.PlateCarree())
    ax_map.set_title(f"{row.nome} - {row.paese}", fontsize=9)
    ax_map.add_feature(cf.LAND, facecolor="lightgray")
    ax_map.add_feature(cf.OCEAN, facecolor="lightblue")
    ax_map.coastlines(linewidth=0.2)
    ax_map.plot(row.lon, row.lat, "ro", markersize=5, transform=ccrs.Geodetic())

    # Serie temporale (destra)
    ax_ts = plt.subplot(n, 2, i * 2 + 2)
    st_id = row["ID"]
    if st_id in data0.columns:
        ax_ts.plot(data0[st_id], color='tab:blue', lw=0.5)
        ax_ts.set_title(f"ID Stazione: {st_id}")
        ax_ts.grid(True, alpha=0.3)
    else:
        ax_ts.text(0.5, 0.5, f"Dati per {st_id} non trovati", ha='center', transform=ax_ts.transAxes)

plt.tight_layout(pad=1.2)
plt.show()

In [ ]:
metadata0 = metadata0.set_index(["ID"])

In [ ]:
x1=data0.index.get_loc('1951-01-31')
x2=data0.index.get_loc('2000-12-31')

# calcolo delle normals, si è fatta la media di tutti i gennaio dal 1950 al 2000 
data_normals = data0[x1:x2].groupby(data0[x1:x2].index.month).mean()
print(data_normals)

In [ ]:
# calcolo vettorializzato delle anomalie (più efficiente e più veloce)
data_anom = data0.copy()
for month in range(1, 13):
    data_anom[data0.index.month == month] -= data_normals.loc[month]

n = len(metadata0)
fig = plt.figure(figsize=(13, 2.5 * n))

for i, (st_id, row) in enumerate(metadata0.iterrows()):
    # Mappa (sinistra)
    ax_map = plt.subplot(n, 2, i * 2 + 1, projection=ccrs.PlateCarree())
    ax_map.set_title(f"{row.nome} - {row.paese}", fontsize=9)
    ax_map.add_feature(cf.LAND, facecolor="lightgray")
    ax_map.add_feature(cf.OCEAN, facecolor="lightblue")
    ax_map.coastlines(linewidth=0.2)
    ax_map.plot(row.lon, row.lat, "ro", markersize=5, transform=ccrs.Geodetic())

    # Serie temporale anomalie (destra)
    ax_ts = plt.subplot(n, 2, i * 2 + 2)
    if st_id in data_anom.columns:
        ax_ts.plot(data_anom[st_id], color='tab:blue', lw=0.8)
        ax_ts.set_title(f"Anomalie: {st_id}")
        ax_ts.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

The anomaly series show the monthly deviation of temperature
from the mean seasonal cycle. A very slightly positive trend is observed at the
most recent stations, consistent with global warming observed
in the 20th and 21st centuries. Interannual variability is high, but the
long-term signal emerges clearly in the most complete series.


In [ ]:
# si crea una griglia utilizzando una risoluzione di 10 gradi
resol = 10
nlon = int(360/resol)
nlat = int(180/resol)


grlons = np.empty([nlon+1],dtype='float')
grlats = np.empty([nlat+1],dtype='float')
grlons[0] = -180.
grlats[0] = -90.

for i in range(1,nlon+1):
    grlons[i]=grlons[i-1]+resol
for i in range(1,nlat+1):
    grlats[i]=grlats[i-1]+resol

In [ ]:
data_mo = np.empty([nmonths,nlat,nlon],dtype=float)
data_mo[:,:,:] = np.nan

data_yr = np.empty([int(nyears),nlat,nlon],dtype=float)
data_yr[:,:,:] = np.nan

In [ ]:
# Iteriamo sulla griglia
for j in range(nlat):
    for i in range(nlon):
        # Filtro rapido combinato delle stazioni dentro la cella (molto più veloce)
        mask = (metadata0.lon >= grlons[i]) & (metadata0.lon < grlons[i+1]) & \
               (metadata0.lat >= grlats[j]) & (metadata0.lat < grlats[j+1])
        dummy = metadata0[mask]

        if not dummy.empty:
            st_ids = dummy.index
            
            # 1. Media mensile allineata a 'taxis' tramite reindex (Fix per ValueError)
            mean_mo = data_anom[st_ids].mean(axis=1).reindex(taxis)
            data_mo[:, j, i] = mean_mo.values
            
            # 2. Media annuale tramite resample
            mean_yr = mean_mo.resample("YE").mean()
            data_yr[:, j, i] = mean_yr.values

            # --- Plotting ottimizzato e abbreviato ---
            fig = plt.figure(figsize=(13, 3))
            
            # Mappa (sinistra)
            ax = fig.add_subplot(1, 2, 1, projection=ccrs.PlateCarree())
            ax.stock_img()
            ax.coastlines()
            ax.gridlines(xlocs=range(-180, 181, resol), ylocs=range(-90, 91, resol), alpha=0.3)
            ax.scatter(dummy.lon, dummy.lat, color='r', marker='o', s=5)
            
            # Serie (destra)
            tser = fig.add_subplot(1, 2, 2)
            tser.plot(taxis, data_anom[st_ids].reindex(taxis), color='grey', lw=0.1, alpha=0.5) 
            tser.plot(taxis, mean_mo, color='blue', lw=0.6) # Media mensile
            tser.plot(mean_yr.index, mean_yr, color='red', lw=2) # Media annuale

            fig.tight_layout()
            plt.show()

The station data were interpolated onto a regular
10° × 10° grid. For each cell containing at least one station, the
monthly (blue) and annual (red) mean of the anomalies are calculated. The annual mean
reduces high-frequency variability and makes the long-term
climate signal more readable. Spatial grid aggregation also allows
a direct comparison with model data.

In [ ]:
fig = plt.figure(figsize=(7, 5))

# Mappa (Top)
ax = fig.add_subplot(2, 1, 1, projection=ccrs.PlateCarree())
ax.stock_img()
ax.coastlines()
ax.gridlines(xlocs=range(-180, 181, resol), ylocs=range(-90, 91, resol))

# Creiamo l'asse temporale degli anni coerente con data_yr (169 anni)
yr_axis = pd.date_range('1850-12-31', periods=int(nyears), freq='YE')

tser = fig.add_subplot(2, 1, 2)

for j in range(nlat):
    for i in range(nlon):
        if not np.isnan(data_yr[:, j, i]).all():
            # Segna la cella sulla mappa
            ax.plot(grlons[i:i+2].sum() / 2, grlats[j:j+2].sum() / 2,
                    color='red', marker='x', markersize=5, transform=ccrs.PlateCarree())
            # Traccia la serie temporale locale nella riga bottom
            tser.plot(yr_axis, data_yr[:, j, i], color='gray', lw=0.1)

# Calcolo vettorializzato della media globale
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='Mean of empty slice')
    data_glob = np.nanmean(data_yr, axis=(1, 2))

# Traccia la media globale
tser.plot(yr_axis, data_glob, color='red', lw=2)
tser.axhline(y=0, color='black', linestyle='-')
tser.set_title('Anomalie di temperatura nel Regno Unito e in Irlanda (media regionale in rosso)') 
tser.set_ylabel('[°C]')

fig.tight_layout()
plt.show()

The graph shows the spatial average of temperature anomalies across all
UK grid cells. A clear positive trend is observed
starting in the second half of the 20th century, with anomaly values
steadily exceeding zero after the 1980s. This signal is
consistent with documented anthropogenic warming on a global scale.

In [ ]:
df1 = pd.DataFrame(pd.date_range('1850', '2015', freq='YE'), columns = ['anno'])
df1 = pd.DataFrame({'anno': yr_axis.year})
df1['anomUKIE'] = data_glob

## Making of the AMO Index.

In [ ]:
modfile=r'C:\Users\kekko\Downloads\Python\Lab of Geo\Amon\psl_Amon_MRI-ESM2-0_historical_r1i1p1f1_gn_185001-201412.nc'

In [ ]:
d1 = d2 = d3 = d4 = xr.open_dataset(modfile)
d1

In [ ]:
# conversione in datetime64
datetimeindex = d1.indexes['time']
datetimeindex = d2.indexes['time']
datetimeindex = d3.indexes['time']
datetimeindex = d4.indexes['time']
d1['time'] = datetimeindex
d2['time'] = datetimeindex
d3['time'] = datetimeindex
d4['time'] = datetimeindex

In [ ]:
# Creazione dei modelli filtrando per latitudine e longitudine
modelloAmo = d1.where((d1.lat > 0) & (d1.lat < 60) & (d1.lon > 285) & (d1.lon < 352.5), drop=True)
modello2   = d2.where((d2.lat < 90) & (d2.lon < 352.5) & (d2.lon <= 285), drop=False)
modello3   = d3.where((d3.lat < 0) & (d3.lon > 290) & (d3.lon < 360), drop=False)
modello4   = d4.where((d4.lat >= 60), drop=False)

modelli = [modelloAmo, modello2, modello3, modello4]
titoli  = ['Modello 1 (AMO)', 'Modello 2', 'Modello 3', 'Modello 4']

# Visualizzazione dei modelli
fig, axes = plt.subplots(2, 2, figsize=(10, 7),
                         subplot_kw={'projection': ccrs.Robinson()})
fig.subplots_adjust(bottom=0.12, hspace=0.05, wspace=0.05)

for ax, modello, titolo in zip(axes.flat, modelli, titoli):
    p = modello.psl.mean('time').plot(ax=ax, transform=ccrs.PlateCarree(),
                                      cmap='jet', extend='both', add_colorbar=False)
    ax.coastlines()
    ax.gridlines()
    ax.set_title(titolo, fontsize=13, fontweight='bold')

cbar_ax = fig.add_axes([0.2, 0.06, 0.6, 0.02])
fig.colorbar(p, cax=cbar_ax, orientation='horizontal', label='PSL (Pa)')

fig.suptitle('Confronto modelli – PSL media temporale', fontsize=15, fontweight='bold')
plt.show()

The map shows the time-averaged pressure of the MRI-ESM2-0 model in the
four defined regions: North Atlantic (AMO zone), the rest of the globe excluding
the North Atlantic, the South Atlantic, and high latitudes. The spatial
division is used to calculate the AMO Index, which is based on the contrast
between the Atlantic signal and that of the rest of the globe.

In [ ]:
# Creazione del modello senza Atlantico unendo i modelli 2, 3 e 4
modelloNoAtl = xr.merge([modello2,modello3,modello4], compat='override')

# Grafico del modello senza Atlantico
p0 = modelloNoAtl.psl.mean('time').plot(transform=ccrs.PlateCarree(),
                                        subplot_kws={'projection': ccrs.Robinson()},
                                        cmap='jet',extend='both')
p0.axes.coastlines()
p0.axes.gridlines()

In [ ]:
# Calcoliamo le normals prendendo il periodo di riferimento, raggruppando per mese e andando a effettuare la media
normalNoAtl = modelloNoAtl.psl.sel(time=slice('1850-01', '1890-12-31')).groupby('time.month').mean()
normalAmo = modelloAmo.psl.sel(time=slice('1850-01', '1890-12-31')).groupby('time.month').mean()

In [ ]:
# Le anomalie vengono calcolate raggruppando per mese le temperature assolute e poi sottraendo le normals
anomalieNoAtl = modelloNoAtl.psl.groupby('time.month') - normalNoAtl
anomalieAmo = modelloAmo.psl.groupby('time.month') - normalAmo

In [ ]:
# modifica dei pesi
# Visto che le aree delle celle diminuiscono con la latitudine si vanno a creare dei pesi proporzionali alle celle
# in modo da poter creare una media pesata
pesiNoAtl = np.cos(np.deg2rad(modelloNoAtl.lat))
pesiAmo = np.cos(np.deg2rad(modelloAmo.lat))

In [ ]:
# Il modello conterrà anomalie di temperature Atlantico del Nord e del globo intero escludendo Nord Atlantico.
# Da una risoluzione mensile passiamo ad una risoluzione annuale, aggregando poi anche nello spazio.
modelloCompleto = xr.Dataset()
modelloCompleto['tNoAtl'] = anomalieNoAtl.weighted(pesiNoAtl).mean(('lon', 'lat'), 
                                                                   keep_attrs=True).resample(time='YE').mean()
modelloCompleto['tAmo'] = anomalieAmo.weighted(pesiAmo).mean(('lon', 'lat'), 
                                                             keep_attrs=True).resample(time='YE').mean()

In [ ]:
# Calcolo della differenza tra le anomalie del modello completo e del modello senza Atlantico
totale = modelloCompleto.tAmo - modelloCompleto.tNoAtl
totale

PSL anomalies are spatially averaged using cosine weights of
latitude, which compensate for the reduction in cell area at high
latitudes. Switching from monthly to annual resolution via resample
further reduces noise. This produces two clean time series:
one for the Atlantic region (tAmo) and one for the rest of the globe (tNoAtl).

In [ ]:
# Costruiamo un dataframe che conterrà anno per anno il valore dell'AMO Index.
df2 = pd.DataFrame(pd.date_range('1850', '2015', freq='YE'), columns = ['anno'])
df2['anno'] = pd.DatetimeIndex(df2['anno']).year
df2['anomAmo'] = totale
df2 = df2.set_index('anno')

In [ ]:
# Grafico dell'AMO Index
# Taglio corretto dell'asse temporale: gli anni vengono presi direttamente
# dall'indice di df2 (1850-2014), invece di un np.linspace fisso non coerente.
plt.figure(figsize=(7,4))
x = df2.anomAmo
y = df2.index.values  # anni effettivi della serie AMO (1850-2014)
plt.axhline(y=0, color='black', linestyle='-')

plt.title('Oscillazione multidecennale atlantica')
plt.xlabel('Anno')
plt.ylabel('AMO Index')

plt.fill_between(y, x, where=x>0, color='orange')
plt.fill_between(y, x, where=x<0, color='blue')

plt.show()

The AMO Index (difference between Atlantic and non-Atlantic anomalies) graph shows alternating positive and negative phases on timescales of 20–40 years, typical of the Atlantic Multidecadal Oscillation.
Positive phases of the AMO are generally associated with higher temperatures in the North Atlantic and Europe. The index calculated here is derived from surface pressure (SSP), not sea surface temperature (SST) as in the classical definition, which can influence the intensity and phase of the signal.

In [ ]:
# Costruiamo un dataframe che conterrà anno per anno il valore dell'AMO Index.
df2.reset_index(inplace = True)
finale = df1.merge(df2, how='left')
finale = finale.set_index('anno')


## Confronto anomalie (Regno Unito + Irlanda) vs AMO Index: scelta del taglio temporale

Il confronto tra la serie delle anomalie di temperatura delle Isole Britanniche
e l'AMO Index richiede una **finestra temporale comune e ben campionata**,
altrimenti il confronto non ha senso fisico:

* l'**AMO Index** è ricavato dal modello MRI-ESM2-0, che copre il periodo
  *1850–2014*: dopo il 2014 non esistono valori da confrontare;
* prima del **1900** le stazioni disponibili nelle Isole Britanniche sono molto
  poche, quindi la media regionale risulta rumorosa e non rappresentativa del
  segnale climatico.

Per questo motivo il confronto viene **ritagliato al periodo 1900–2014**, dove
entrambe le serie sono affidabili. Si usa inoltre una media mobile **centrata** a
15 anni per isolare il segnale multidecennale (l'AMO ha periodi tipici di 20–40
anni) ed eliminare la variabilità interannuale, rendendo il confronto visivamente
leggibile e logicamente coerente.

In [ ]:
# Confronto tra le anomalie di temperatura (Regno Unito + Irlanda) e l'AMO Index.
# Il confronto viene ristretto a una finestra temporale comune e ben campionata:
#  - l'AMO Index derivato dal modello termina nel 2014 (nessun dato dopo);
#  - prima del 1900 le stazioni disponibili nelle Isole Britanniche sono pochissime,
#    quindi la media regionale e' rumorosa e poco rappresentativa.
# Si seleziona quindi il periodo 1900-2014, dove entrambe le serie sono affidabili.
START, END = 1900, 2014
conf = finale.loc[START:END]

plt.figure(figsize=(10, 4))

# media mobile CENTRATA a 15 anni: isola il segnale multidecennale (l'AMO ha
# periodi tipici di 20-40 anni) eliminando la variabilita' interannuale.
ax3 = conf.anomAmo.rolling(window=15, center=True).mean().plot(
    color='green', grid=True, label='AMO Index')
ax4 = conf.anomUKIE.rolling(window=15, center=True).mean().plot(
    color='red', grid=True, secondary_y=True, label='Anomalie UK+IRL')

plt.title('Confronto tra anomalie di temperatura (Regno Unito + Irlanda) e AMO Index (1900-2014)')
ax3.set_xlabel('Anno')
ax3.set_ylabel('AMO Index', color='green')
ax4.set_ylabel('Anomalie UK+IRL [°C]', color='red')

h1, l1 = ax3.get_legend_handles_labels()
h2, l2 = ax4.get_legend_handles_labels()
ax3.legend(h1 + h2, l1 + l2, loc='upper left')

plt.show()

## Osservazioni sul confronto

Confrontando le due serie nel periodo 1900–2014 si osserva come le anomalie di
temperatura delle Isole Britanniche (Regno Unito + Irlanda) tendano a seguire le
fasi multidecennali dell'AMO Index: alle fasi positive dell'AMO corrispondono in
genere periodi relativamente più caldi nell'area, mentre alle fasi negative
periodi più freddi. Va sottolineato che l'AMO calcolato qui deriva dalla
pressione al livello del mare (PSL) del modello e non dalla temperatura
superficiale del mare (SST) come nella definizione classica: questo può attenuare
o sfasare la corrispondenza tra i due segnali. Sovrapposto alla modulazione
dell'AMO resta comunque visibile un **trend di fondo di riscaldamento** nella
seconda metà del XX secolo, coerente con il riscaldamento globale di origine
antropica.